# Paper 3 · Phase 1a — DroneDetect exploration

DroneDetect V2 (Swinney & Woods): 7 drones, Nuand bladeRF, complex I/Q at 60 MS/s, centre 2.4375 GHz, 28 MHz bandwidth.
Folders `{CLEAN,BLUE,WIFI,BOTH}/{DRONE}_{MODE}/`, files `{PFX}_{iiMM}_{k}.dat` (interleaved float32 I/Q):
`ii` = interference (00 clean, 01 Bluetooth, 10 Wi-Fi, 11 both), `MM` = mode (00 on, 01 hover, 10 fly), `k` = recording index.

Questions: inventory and balance; signal format and level; how bursty each drone is (chunking strategy); how visible the
interference is (shift A strength); whether drone-free recordings exist (Background class). Reads only small slices; nothing is written
except figures in `results/paper3/phase1/explore/`.

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, welch
from dotenv import load_dotenv

load_dotenv()
DD = os.getenv("DRONEDETECT_PATH")
fig_path = os.path.join(os.getenv("DATA_PATH"), "results/paper3/phase1/explore")
os.makedirs(fig_path, exist_ok=True)

FS, FC = 60e6, 2.4375e9
DRONES = {"AIR": "DJI Mavic 2 Air S", "DIS": "Parrot Disco", "INS": "DJI Inspire 2", "MIN": "DJI Mavic Mini",
          "MP1": "DJI Mavic Pro", "MP2": "DJI Mavic Pro 2", "PHA": "DJI Phantom 4"}
INTERF = {"00": "CLEAN", "01": "BLUE", "10": "WIFI", "11": "BOTH"}
MODES = {"00": "ON", "01": "HO", "10": "FY"}

## Inventory

In [ ]:
rows = []
for p in sorted(glob.glob(os.path.join(DD, "*", "*", "*.dat"))):
    subset, folder, name = p.split(os.sep)[-3:]
    drone, mode_dir = folder.split("_")
    pfx, code_, k = re.fullmatch(r"(\w{3})_([01]{4})_(\d{2})\.dat", name).groups()
    assert INTERF[code_[:2]] == subset and MODES[code_[2:]] == mode_dir, p
    rows.append({"path": p, "subset": subset, "drone": drone, "prefix": pfx, "mode": mode_dir, "k": int(k),
                 "n_samples": os.path.getsize(p) // 8})
inv = pd.DataFrame(rows)
inv["duration_s"] = inv["n_samples"] / FS
print(len(inv), "files,", inv["n_samples"].sum() * 8 / 1e9, "GB")
print(inv.groupby("drone")["prefix"].unique())
inv.pivot_table(index="drone", columns=["subset", "mode"], values="k", aggfunc="count", fill_value=0)

In [ ]:
inv["duration_s"].describe(), inv.loc[inv["duration_s"] < 1.99, ["subset", "drone", "mode", "k", "duration_s"]]

## Signal level, DC offset and burstiness (1 M samples from the middle of every file)

In [ ]:
def mid_slice(path, n=1_000_000):
    m = np.memmap(path, dtype=np.complex64, mode="r")
    s = len(m) // 2 - n // 2
    return np.asarray(m[s:s + n])

def stats(x):
    dc = x.mean()
    p = np.abs(x - dc) ** 2
    p_win = p[: len(p) // 4000 * 4000].reshape(-1, 4000).mean(1)  # 4000-sample windows (67 µs)
    floor = np.percentile(p_win, 10)
    return {"dc_abs": abs(dc), "rms": np.sqrt(p.mean()), "peak": np.sqrt(p.max()),
            "papr_db": 10 * np.log10(p.max() / p.mean()),
            "active_frac": (p_win > 4 * floor).mean(),  # windows 6 dB above the quiet floor
            "dyn_db": 10 * np.log10(np.percentile(p_win, 99) / floor)}

st = pd.DataFrame([{**r, **stats(mid_slice(r["path"]))} for r in inv.to_dict("records")])
st.groupby(["drone", "subset"])[["dc_abs", "rms", "active_frac", "dyn_db"]].mean().unstack("subset").round(3)

## Spectrograms: every drone, CLEAN, switched on (first 20 ms)

In [ ]:
def spec(ax, path, ms=20, nfft=1024):
    x = np.asarray(np.memmap(path, dtype=np.complex64, mode="r")[: int(FS * ms / 1000)])
    f, t, S = spectrogram(x - x.mean(), fs=FS, nperseg=nfft, noverlap=0, return_onesided=False, mode="psd")
    f, S = np.fft.fftshift(f), np.fft.fftshift(S, axes=0)
    ax.pcolormesh(t * 1e3, (FC + f) / 1e9, 10 * np.log10(S + 1e-20), shading="auto", cmap="viridis",
                  vmin=np.percentile(10 * np.log10(S + 1e-20), 5), vmax=np.percentile(10 * np.log10(S + 1e-20), 99.9))
    ax.set_xlabel("ms"); ax.set_ylabel("GHz")

def first(subset, drone, mode):
    return inv[(inv.subset == subset) & (inv.drone == drone) & (inv["mode"] == mode)].sort_values("k").iloc[0]["path"]

fig, axes = plt.subplots(2, 4, figsize=(22, 8)); axes = axes.ravel()
for ax, d in zip(axes, DRONES):
    spec(ax, first("CLEAN", d, "ON")); ax.set_title(f"{d} · {DRONES[d]} · CLEAN ON")
axes[-1].axis("off")
plt.tight_layout(); fig.savefig(os.path.join(fig_path, "spec_clean_on.png"), dpi=110); plt.show()

## Same drone across interference subsets (hover)

In [ ]:
for d in ["MIN", "PHA"]:
    fig, axes = plt.subplots(1, 4, figsize=(22, 4))
    for ax, s in zip(axes, INTERF.values()):
        spec(ax, first(s, d, "HO")); ax.set_title(f"{d} · {s} · HO")
    plt.tight_layout(); fig.savefig(os.path.join(fig_path, f"spec_interf_{d}.png"), dpi=110); plt.show()

## Average PSD per drone, CLEAN vs BOTH (Welch over the middle 1 M samples, all modes and recordings)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 7), sharex=True); axes = axes.ravel()
for ax, d in zip(axes, DRONES):
    for s in ["CLEAN", "BOTH"]:
        P = []
        for p in inv[(inv.drone == d) & (inv.subset == s)]["path"]:
            x = mid_slice(p); f, Pxx = welch(x - x.mean(), fs=FS, nperseg=1024, return_onesided=False); P.append(Pxx)
        f, P = np.fft.fftshift(f), np.fft.fftshift(np.mean(P, 0))
        ax.plot((FC + f) / 1e9, 10 * np.log10(P), label=s, lw=0.8)
    ax.set_title(d); ax.legend(); ax.set_xlabel("GHz")
axes[-1].axis("off")
plt.tight_layout(); fig.savefig(os.path.join(fig_path, "psd_clean_vs_both.png"), dpi=110); plt.show()

## Findings

**Inventory.**
- 390 files, 374 GB.
- Each (subset, drone, mode) has 5 recordings of 2 s; two INS FY files are shorter (1.76 s and 1.82 s).
- Gaps:
  - DIS has no HO mode (the Disco is fixed-wing and can't hover);
  - PHA FY is missing in CLEAN and BLUE, so Phantom fly mode exists only under interference. Exclude PHA FY from the shift-A comparison, or report per mode.
- Folder `MP1` holds files prefixed `MA1`, and `MP2` holds `MAV`; the folder name is the label.
- Code `iiMM`: `ii` = interference, `MM` = mode (00 on, 01 hover, 10 fly). Every file name matches its folder.

**No drone-free recordings.** DroneDetect has no Background class.

**Two receiver configurations (shortcut risk).** The data splits into two groups that differ in more than the drone:

| | AIR, DIS, PHA | INS, MIN, MP1, MP2 |
|---|---|---|
| DC offset \|mean\| | 0.034–0.065 | 0.008–0.010 |
| RMS | 0.07–0.20 | 0.004–0.021 |
| out-of-band floor | ≈ −110…−120 dB, raised 35 MHz plateau | ≈ −140 dB flat, steep filter edges |
| file length | exactly 1.2e8 samples | 1.2e8 + ~4,600 |
| gain steps inside a file | yes (DIS ~5.5 ms, PHA ~4 ms) | not seen |

A classifier can tell the groups apart from the noise floor alone. Phantom 4 shows little structure of its own: its CLEAN files are dominated by the raised plateau.

**Interference shift is confounded with a gain change.** For AIR and PHA, CLEAN vs BOTH also changes the noise-floor level by up to 20 dB. Shift A is therefore "interference + session/gain change", not interference alone. This is realistic for the paper, but it must be stated.

**Signals.**
- MIN: narrow (~2 MHz) bursts near 2.437 GHz.
- MP1/MP2: ~18–20 MHz OcuSync-like video bursts.
- INS: ~10 MHz hopping bursts.
- DIS: short wideband bursts.
- Bursty drones occupy only about 15–55% of 4000-sample windows (6 dB above the quiet floor). A fixed-grid chunking would label empty noise windows as drones.
